# Video → 3D (VGGT on Colab)

This notebook:
1. Installs **VGGT**
2. Loads weights from **Hugging Face** (`facebook/VGGT-1B`)
3. Accepts your **video**
4. Exports a **GLB** you can upload back into the 3D Reconstruction app

**Runtime → Change runtime type → T4 GPU** (or better) before running.


In [ ]:
# 1) Install dependencies (GPU runtime)
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip -q install numpy opencv-python trimesh huggingface_hub einops Pillow tqdm scipy matplotlib


In [ ]:
# 2) Clone VGGT and install
import os
if not os.path.isdir("vggt"):
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git
%cd vggt
!pip -q install -e .
%cd /content


In [ ]:
# 3) Upload your indoor / phone video
from google.colab import files
uploaded = files.upload()
assert uploaded, "Upload one video file"
VIDEO_PATH = list(uploaded.keys())[0]
print("Using video:", VIDEO_PATH)


In [ ]:
# 4) Sample frames from the video (keep it small for Colab free tier)
import cv2
from pathlib import Path

frames_dir = Path("frames")
frames_dir.mkdir(exist_ok=True)
for p in frames_dir.glob("*"):
    p.unlink()

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
# ~1 frame per second, max 40 frames (fits free Colab memory better)
interval = max(1, int(round(fps)))
max_frames = 40
idx = saved = 0
while saved < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    if idx % interval == 0:
        out = frames_dir / f"{saved:06d}.jpg"
        cv2.imwrite(str(out), frame)
        saved += 1
    idx += 1
cap.release()
image_names = sorted(str(p) for p in frames_dir.glob("*.jpg"))
print(f"Extracted {len(image_names)} frames")
assert len(image_names) >= 2, "Need at least 2 frames — try a longer clip"


In [ ]:
# 5) Load VGGT from Hugging Face and run reconstruction
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Enable a GPU runtime: Runtime → Change runtime type → GPU"

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()

images = load_and_preprocess_images(image_names).to(device)
print("images tensor:", tuple(images.shape))

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions = model(images)

# Move tensors to CPU numpy-friendly form where needed by export helpers
print("keys:", sorted(predictions.keys()))


In [ ]:
# 6) Export GLB
# VGGT ships a helper in demo scripts; fall back to a small export if import paths differ.
import numpy as np
import trimesh

try:
    from vggt.utils.geometry import unproject_depth_map_to_point_map
except Exception:
    unproject_depth_map_to_point_map = None

# Prefer world points if present
if "world_points" in predictions:
    pts = predictions["world_points"]
    conf = predictions.get("world_points_conf", None)
elif "world_points_from_depth" in predictions:
    pts = predictions["world_points_from_depth"]
    conf = predictions.get("depth_conf", None)
else:
    raise RuntimeError(f"Unexpected prediction keys: {list(predictions)}")

def to_np(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x)

pts = to_np(pts)
# shapes are often (S,H,W,3) or (1,S,H,W,3)
pts = pts.reshape(-1, 3)

colors = None
if "images" in predictions:
    imgs = to_np(predictions["images"])
    # (S,3,H,W) or similar → match point count roughly by resize flatten
    if imgs.ndim == 5:
        imgs = imgs[0]
    if imgs.ndim == 4 and imgs.shape[1] in (1, 3):
        # S,C,H,W → S,H,W,C
        imgs = np.transpose(imgs, (0, 2, 3, 1))
    colors = (np.clip(imgs.reshape(-1, imgs.shape[-1]), 0, 1) * 255).astype(np.uint8)
    if colors.shape[0] != pts.shape[0]:
        colors = None

# Confidence filter
if conf is not None:
    conf = to_np(conf).reshape(-1)
    if conf.shape[0] == pts.shape[0]:
        thr = np.percentile(conf, 50)
        keep = conf >= thr
        pts = pts[keep]
        if colors is not None:
            colors = colors[keep]

# Subsample for a lighter GLB
max_points = 200_000
if pts.shape[0] > max_points:
    sel = np.random.default_rng(0).choice(pts.shape[0], max_points, replace=False)
    pts = pts[sel]
    if colors is not None:
        colors = colors[sel]

if colors is None:
    cloud = trimesh.points.PointCloud(vertices=pts)
else:
    cloud = trimesh.points.PointCloud(vertices=pts, colors=colors)

scene = trimesh.Scene(cloud)
out_path = "/content/scene.glb"
scene.export(out_path)
print("Wrote", out_path, "points:", pts.shape[0])


In [ ]:
# 7) Download the GLB, then upload it in your Streamlit app for the same job
from google.colab import files
files.download("/content/scene.glb")
print("In the app: open your job → Upload GLB → Download 3D")
